# 03 Fault Testing

Inject faults into the same sensors and actuators used by the PLC simulation.

In [ ]:
# Colab bootstrap: install only free packages and load this repository.
!pip -q install numpy pandas matplotlib scipy
from pathlib import Path
import subprocess, sys
REPO_URL = 'https://github.com/SaadWajih99/industrial-conveyor-plc-hmi'
PROJECT = Path('industrial-conveyor-plc-hmi')
if not (PROJECT / 'simulation').exists():
    if 'REPLACE_WITH_GITHUB_USERNAME' in REPO_URL:
        raise RuntimeError('Set REPO_URL to the public GitHub repository URL before running this notebook.')
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT)], check=True)
sys.path.insert(0, str(PROJECT.resolve()))


In [ ]:
from simulation.testbench import ScenarioConfig, run_scenario
scenarios = {
    'motor_failure': ScenarioConfig(duration_s=3, arrivals=(), motor_failure=True),
    'entry_stuck_on': ScenarioConfig(duration_s=3, arrivals=(), sensor_faults={'entry_sensor': 'stuck_on'}),
    'entry_stuck_off': ScenarioConfig(duration_s=12, arrivals=((1.0, False),), sensor_faults={'entry_sensor': 'stuck_off'}),
    'diverter_timeout': ScenarioConfig(duration_s=12, arrivals=((1.0, True),), diverter_failure=True),
    'product_timeout': ScenarioConfig(duration_s=22, arrivals=((1.0, False),), sensor_faults={'exit_sensor': 'stuck_off'}),
    'impossible_limits': ScenarioConfig(duration_s=2, arrivals=(), force_diverter_both_limits=True),
}
for name, config in scenarios.items():
    result = run_scenario(config)
    print(name, result.final_plc['fault_id'], result.final_plc['fault_message'], 'safe=', not result.records[-1]['conveyor_motor'])


A physical installation would additionally validate wiring, diagnostic coverage and safety response independently of the standard PLC program.